<div style="text-align: center;">
  <h1><b>🧠 Brain Tumor Detection using YOLOv8n 🏥</b></h1>
  <p><i>High-Speed Deep Learning Model for Accurate Tumor Localization in MRI Scans</i></p>
</div>

<p align="center">
  <img src="1_KNjung7q4s4V5U_ypVsVZA.jpg" width="800"/>
</p>

# **Our Details**

### ‎‧₊˚✿[My Name]✿˚ : **Mohamed Reda Ramadan Khamis**
### ‎‧₊˚✿[Phone Number]✿˚: **01554725661**

In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od
od.download ("https://www.kaggle.com/datasets/ahmedhamada0/brain-tumor-detection")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: Mohamed Reda
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/ahmedhamada0/brain-tumor-detection


100%|██████████| 84.0M/84.0M [00:02<00:00, 30.7MB/s]


In [ ]:
!pip install ultralytics opencv-python kaggle -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.4 MB/s eta 0:00:00


In [ ]:
import os
import json
import cv2
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
DATA_DIR = "/content/brain-tumor-detection/Br35H-Mask-RCNN"
OUTPUT_DIR = "/content/yolo_dataset"

os.makedirs(f"{OUTPUT_DIR}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/images/val", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/val", exist_ok=True)

In [ ]:
with open(os.path.join(DATA_DIR, "annotations_all.json")) as f:
    data = json.load(f)

images = data # data itself is the dictionary of images

annotations = []
for image_id, image_info in data.items():
    if "regions" in image_info:
        for region in image_info["regions"]:
            # Add image_id to the region for context
            region['image_id'] = image_id
            annotations.append(region)

In [ ]:
def convert_bbox(size, bbox):
    dw = 1. / size[0]
    dh = 1. / size[1]

    x = bbox[0] + bbox[2] / 2.0
    y = bbox[1] + bbox[3] / 2.0
    w = bbox[2]
    h = bbox[3]

    x *= dw
    y *= dh
    w *= dw
    h *= dh

    return (x, y, w, h)

In [ ]:
img_ids = list(images.keys())
train_ids, val_ids = train_test_split(img_ids, test_size=0.2, random_state=42)

for ann in annotations:
    img_id = ann["image_id"]
    img_info = images[img_id]

    file_name = img_info["filename"]

    img_path = os.path.join(DATA_DIR, "TRAIN", file_name)

    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]

    shape_type = ann["shape_attributes"]["name"]
    bbox = []

    if shape_type == "polygon":
        points_x = ann["shape_attributes"]["all_points_x"]
        points_y = ann["shape_attributes"]["all_points_y"]

        x_min = min(points_x)
        y_min = min(points_y)
        x_max = max(points_x)
        y_max = max(points_y)

        bbox_x = x_min
        bbox_y = y_min
        bbox_w = x_max - x_min
        bbox_h = y_max - y_min
        bbox = [bbox_x, bbox_y, bbox_w, bbox_h]
    elif shape_type == "ellipse":
        cx = ann["shape_attributes"]["cx"]
        cy = ann["shape_attributes"]["cy"]
        rx = ann["shape_attributes"]["rx"]
        ry = ann["shape_attributes"]["ry"]

        bbox_x = cx - rx
        bbox_y = cy - ry
        bbox_w = 2 * rx
        bbox_h = 2 * ry
        bbox = [bbox_x, bbox_y, bbox_w, bbox_h]
    elif shape_type == "circle":
        cx = ann["shape_attributes"]["cx"]
        cy = ann["shape_attributes"]["cy"]
        r = ann["shape_attributes"]["r"]

        bbox_x = cx - r
        bbox_y = cy - r
        bbox_w = 2 * r
        bbox_h = 2 * r
        bbox = [bbox_x, bbox_y, bbox_w, bbox_h]
    # Add other shape types (e.g., 'rect') if present in your data
    else:
        # Skip annotations with unhandled shape types for now
        print(f"Skipping unhandled shape type: {shape_type}")
        continue

    yolo_bbox = convert_bbox((w, h), bbox)

    split = "train" if img_id in train_ids else "val"

    # Copy image
    out_img_path = f"{OUTPUT_DIR}/images/{split}/{file_name}"
    cv2.imwrite(out_img_path, img)

    # Create label file
    label_path = f"{OUTPUT_DIR}/labels/{split}/{file_name.replace('.jpg', '.txt')}"

    with open(label_path, "a") as f:
        f.write(f"0 {' '.join(map(str, yolo_bbox))}\n")

In [ ]:
data_yaml = """
path: /content/yolo_dataset

train: images/train
val: images/val

names:
  0: tumor
"""

with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="brain_tumor_yolo"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, i

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78be0e639c70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
model.val()

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1613.2±460.9 MB/s, size: 42.4 KB)
val: Scanning /content/yolo_dataset/labels/val.cache... 94 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 94/94 39.4Mit/s 0.0s
val: /content/yolo_dataset/images/val/y10.jpg: 3 duplicate labels removed
val: /content/yolo_dataset/images/val/y101.jpg: 2 duplicate labels removed
val: /content/yolo_dataset/images/val/y109.jpg: 2 duplicate labels removed
val: /content/yolo_dataset/images/val/y110.jpg: 2 duplicate labels removed
val: /content/yolo_dataset/images/val/y118.jpg: 2 duplicate labels removed
val: /content/yolo_dataset/images/val/y120.jpg: 2 duplicate labels removed
val: /content/yolo_dataset/images/val/y137.jpg: 2 duplicate labels removed
val: /content/yolo_dataset/images/val/y139.jpg: 2 duplicate labels removed
val: /content/yolo_dat

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78bd0ba18770>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
results = model.predict(
    source="/content/yolo_dataset/images/val",
    conf=0.25,
    save=True
)


image 1/94 /content/yolo_dataset/images/val/y10.jpg: 640x576 (no detections), 39.1ms
image 2/94 /content/yolo_dataset/images/val/y101.jpg: 640x576 2 tumors, 7.1ms
image 3/94 /content/yolo_dataset/images/val/y109.jpg: 640x640 2 tumors, 8.0ms
image 4/94 /content/yolo_dataset/images/val/y110.jpg: 640x576 1 tumor, 8.1ms
image 5/94 /content/yolo_dataset/images/val/y118.jpg: 640x640 1 tumor, 8.8ms
image 6/94 /content/yolo_dataset/images/val/y120.jpg: 640x544 2 tumors, 45.5ms
image 7/94 /content/yolo_dataset/images/val/y137.jpg: 640x544 1 tumor, 7.2ms
image 8/94 /content/yolo_dataset/images/val/y139.jpg: 640x480 1 tumor, 43.6ms
image 9/94 /content/yolo_dataset/images/val/y155.jpg: 640x480 1 tumor, 6.3ms
image 10/94 /content/yolo_dataset/images/val/y168.jpg: 640x512 1 tumor, 43.8ms
image 11/94 /content/yolo_dataset/images/val/y174.jpg: 640x544 1 tumor, 7.7ms
image 12/94 /content/yolo_dataset/images/val/y192.jpg: 640x576 1 tumor, 7.8ms
image 13/94 /content/yolo_dataset/images/val/y198.jpg: 640

In [ ]:
import shutil

BEST = "/content/runs/detect/brain_tumor_yolo/weights/best.pt"
shutil.copy(BEST, "/content/Yolov8_best_model.pt")

'/content/Yolov8_best_model.pt'